<a href="https://colab.research.google.com/github/DrewThomasson/Derme---Cosmetic-Wellness-Platform/blob/main/notebooks/Derme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🏥 Derme - Cosmetic Wellness Platform - Google Colab Setup
# FINAL FIX: Matched Port 7860

import os
import subprocess
import sys
import time
import threading
import re
import socket
from tqdm.notebook import tqdm

print("=" * 70)
print("🏥 DERME - COSMETIC WELLNESS PLATFORM")
print("   Ingredient Scanner & Allergen Tracker")
print("=" * 70)
print()

# Configuration
REPO_URL = "https://github.com/DrewThomasson/Derme---Cosmetic-Wellness-Platform"
REPO_NAME = "Derme---Cosmetic-Wellness-Platform"
# MATCHED PORT: The app hardcodes 7860, so we must use it.
PORT = 7860

def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# Step 1: Install Tesseract OCR
print("📦 Step 1/5: Installing Tesseract OCR...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "libtesseract-dev"], capture_output=True)
print("   ✅ Tesseract OCR installed!\n")

# Step 2: Clone Repository
print("📂 Step 2/5: Cloning repository...")
os.chdir("/content")
if os.path.exists(REPO_NAME):
    subprocess.run(["rm", "-rf", REPO_NAME], capture_output=True)

subprocess.run(["git", "clone", REPO_URL], capture_output=True)
print("   ✅ Repository cloned!\n")

# Step 3: Install Python dependencies
os.chdir(f"/content/{REPO_NAME}")
print("📚 Step 3/5: Installing Python packages...")

# Install requirements
subprocess.run(["pip", "install", "-r", "requirements.txt"], capture_output=True)
print("   ✅ Python packages installed!\n")

# Step 4: Install cloudflared
print("🌐 Step 4/5: Installing Cloudflare Tunnel...")
os.chdir("/content")
if not os.path.exists("cloudflared-linux-amd64.deb"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], capture_output=True)
    subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print("   ✅ Cloudflare Tunnel installed!\n")

# Step 5: Start App
os.chdir(f"/content/{REPO_NAME}")
print("🚀 Step 5/5: Starting Derme app...")

# Create a log file to capture Flask errors
log_file = open("flask_output.log", "w")

def run_flask():
    env = os.environ.copy()
    cmd = ["python", "app.py"]
    subprocess.Popen(
        cmd,
        stdout=log_file,
        stderr=log_file,
        text=True,
        env=env
    )

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

# HEALTH CHECK LOOP
# We wait for port 7860 to become active
print(f"⏳ Waiting for Flask to initialize (checking port {PORT})...")
app_ready = False
for i in range(45): # Wait up to 45 seconds (app takes a moment to load allergens)
    if is_port_in_use(PORT):
        app_ready = True
        break
    time.sleep(1)

if not app_ready:
    print("\n❌ ERROR: The Flask app failed to start.")
    print("   Here is the error log from the app:\n")
    print("-" * 50)
    with open("flask_output.log", "r") as f:
        print(f.read())
    print("-" * 50)
    raise Exception("App failed to start - check logs above")

print(f"   ✅ App is running on localhost:{PORT}!\n")

# Start Tunnel
print("🌐 Creating public tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(3)
public_url = None

# Extract URL with a timeout
for i in range(20):
    line = tunnel_process.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            break
    time.sleep(0.5)

print("\n" + "=" * 70)
if public_url:
    print("🎉 SUCCESS! YOUR APP IS ONLINE:")
    print(f"\n   👉 {public_url} \n")
else:
    print("⚠️  Could not auto-extract URL. Check the logs below:")
    print(tunnel_process.stderr.read())

print("=" * 70)
print("📝 Logs (Keeping alive...):")

try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Stopping...")
    tunnel_process.terminate()
    log_file.close()

🏥 DERME - COSMETIC WELLNESS PLATFORM
   Ingredient Scanner & Allergen Tracker

📦 Step 1/5: Installing Tesseract OCR...
   ✅ Tesseract OCR installed!

📂 Step 2/5: Cloning repository...
   ✅ Repository cloned!

📚 Step 3/5: Installing Python packages...
   ✅ Python packages installed!

🌐 Step 4/5: Installing Cloudflare Tunnel...
   ✅ Cloudflare Tunnel installed!

🚀 Step 5/5: Starting Derme app...
⏳ Waiting for Flask to initialize (checking port 7860)...
